In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch

/usr/local/lib/python3.9/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#查看训练器版本

In [2]:
import transformers
print(transformers.__version__)

4.42.3


# 加载预训练模型和分词器
如果速度慢，修改指向
export HF_ENDPOINT=https://hf-mirror.com

In [3]:
model_name = "defog/llama-3-sqlcoder-8b"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto", #启用了DeepSpeed 不再需要
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.83s/it]
/usr/local/lib/python3.9/dist-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


# 在初始化完model和tokenizer后 设置 pad_token

将 eos_token 作为填充标记，确保训练过程中可以正确处理填充。

In [4]:
# 设置填充标记为 eos_token
tokenizer.pad_token = tokenizer.eos_token

# 如果调整了特殊标记，更新模型的词汇表大小
model.resize_token_embeddings(len(tokenizer))

Embedding(128256, 4096)

# 加载微调数据集

先安装dataset 
pip install datasets

In [5]:
from datasets import load_dataset
#load_dataset 默认会尝试从 Hugging Face Hub 上加载数据集
#dataset = load_dataset("train_test.json")

#本地加载数据集
# 指定本地文件路径
dataset = load_dataset("json", data_files="train_test.json")

In [6]:
train_model_name = "../train_model/kpaas_train_model"

#清理GPU缓存

In [7]:
# Step 2: 清理 GPU 缓存
# torch.cuda.empty_cache()

# 使用 DeepSpeed 的 ZeRO-Offload
DeepSpeed 支持将参数、梯度或优化器状态溢出到 CPU 或磁盘，从而缓解显存不足问题。

In [8]:
# 定义 DeepSpeed 配置文件路径
ds_config = "deepspeed_config.json"

# 定义训练参数

安装deepspeed
pip install deepspeed

如果安装失败，则确保确保 setuptools 和 pip 是最新版本：
pip install --upgrade pip setuptools packaging

然后再重新安装
pip install deepspeed


In [9]:
training_args = TrainingArguments(
    output_dir=train_model_name,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=5000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=1000,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True  # 如果有 GPU，开启混合精度训练
    #,deepspeed=ds_config,  # DeepSpeed 配置
    #local_rank=-1  # 禁用分布式训练,启用单机训练
)

# 检查模型是否已被部分加载到 CPU 或磁盘


使用 accelerate 加载模型时，确保模型未被部分卸载。如果您需要显式加载到 GPU，可以在创建 Trainer 之前执行以下操作：

In [10]:
from accelerate import init_empty_weights, load_checkpoint_and_dispatch
from accelerate import Accelerator
import os



# TrainingArguments 中配置了 deepspeed，Trainer 会自动使用 Accelerator 初始化并将其与 DeepSpeed 配合使用。
#如果启用deepspeed，则这里不用显示初始化accelerator

# 初始化 Accelerator
accelerator = Accelerator()

# 初始化模型
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

#定义权重文件夹路径（model-00001-of-00004.safetensors这些就是项目的权重文件）
path_to_checkpoint=os.path.expanduser("~/.cache/huggingface/hub/models--defog--llama-3-sqlcoder-8b/snapshots/0f96d32e16737bda1bbe0d8fb13a932a8a3fa0bb/")

#定义卸载模块路径，模型的子模块会被卸载到该文件夹，节省一部分内存。
offload_folder="/mnt/workspace/sqlcoder/train/offload/"
    
# 从保存点加载并分发
model = load_checkpoint_and_dispatch(
    model, 
    path_to_checkpoint, 
    device_map="auto", #如果启用了DeepSpeed 不再需要
    offload_folder=offload_folder  # 添加 offload_folder 参数
)



Detected kernel version 4.19.24, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 16.06it/s]
/usr/local/lib/python3.9/dist-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or uns

# （停用！停用）创建 Trainer
原因： 上面load_checkpoint_and_dispatch使用了offload_folder解决内存问题，使得部分模块被卸载到磁盘，会导致Trainer 无法正确处理。

<!-- # 使用 Accelerator 调度模型
# model = accelerator.prepare(model)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     tokenizer=tokenizer,
# ) -->

In [11]:
print(dataset["train"][0])  # 替换 "train" 为你的数据集分区名称

{'instruction': 'Answer the question based on the schema provided.', 'context': "### Database Schema\n CREATE TABLE `base_module`( `F_Id` varchar(50) NOT NULL, `F_ParentId` varchar(50) DEFAULT NULL, `F_Type` int(11) DEFAULT NULL COMMENT '功能类别', `F_FullName` varchar(50) DEFAULT NULL, `F_EnCode` varchar(50) DEFAULT NULL, `F_UrlAddress` longtext, `F_IsButtonAuthorize` int(11) DEFAULT NULL COMMENT '按钮权限', `F_IsColumnAuthorize` int(11) DEFAULT NULL COMMENT '列表权限', `F_IsDataAuthorize` int(11) DEFAULT NULL COMMENT '数据权限', `F_PropertyJson` longtext, `F_Description` longtext, `F_SortCode` bigint(20) DEFAULT NULL COMMENT '排序', `F_EnabledMark` int(11) DEFAULT NULL COMMENT '有效标志', `F_CreatorTime` datetime DEFAULT NULL COMMENT '创建时间', `F_CreatorUserId` varchar(50) DEFAULT NULL, `F_LastModifyTime` datetime DEFAULT NULL COMMENT '修改时间', `F_LastModifyUserId` varchar(50) DEFAULT NULL, `F_DeleteMark` int(11) DEFAULT NULL COMMENT '删除标志', `F_DeleteTime` datetime DEFAULT NULL COMMENT '删除时间', `F_DeleteUserId

# 创建Trainer前先做数据的预处理

In [12]:
# 数据预处理
# def preprocess_function(examples):
#     encoded = tokenizer(examples["text"], padding="max_length", truncation=True)
#     encoded["labels"] = encoded["input_ids"].copy()  # 添加 labels
#     return encoded

def preprocess_function(examples):
    # 将批量的字段拼接起来
    texts = [
        instruction + " " + context + " " + input_text + " " + output_text
        for instruction, context, input_text, output_text in zip(
            examples["instruction"], examples["context"], examples["input"], examples["output"]
        )
    ]
    # 对每条拼接的文本进行编码
    encoded = tokenizer(texts, padding="max_length", truncation=True)
    # 添加 labels
    encoded["labels"] = encoded["input_ids"]
    return encoded

encoded_dataset = dataset.map(preprocess_function, batched=True)

# 检查数据集结构
print(encoded_dataset["train"][0])


{'instruction': 'Answer the question based on the schema provided.', 'context': "### Database Schema\n CREATE TABLE `base_module`( `F_Id` varchar(50) NOT NULL, `F_ParentId` varchar(50) DEFAULT NULL, `F_Type` int(11) DEFAULT NULL COMMENT '功能类别', `F_FullName` varchar(50) DEFAULT NULL, `F_EnCode` varchar(50) DEFAULT NULL, `F_UrlAddress` longtext, `F_IsButtonAuthorize` int(11) DEFAULT NULL COMMENT '按钮权限', `F_IsColumnAuthorize` int(11) DEFAULT NULL COMMENT '列表权限', `F_IsDataAuthorize` int(11) DEFAULT NULL COMMENT '数据权限', `F_PropertyJson` longtext, `F_Description` longtext, `F_SortCode` bigint(20) DEFAULT NULL COMMENT '排序', `F_EnabledMark` int(11) DEFAULT NULL COMMENT '有效标志', `F_CreatorTime` datetime DEFAULT NULL COMMENT '创建时间', `F_CreatorUserId` varchar(50) DEFAULT NULL, `F_LastModifyTime` datetime DEFAULT NULL COMMENT '修改时间', `F_LastModifyUserId` varchar(50) DEFAULT NULL, `F_DeleteMark` int(11) DEFAULT NULL COMMENT '删除标志', `F_DeleteTime` datetime DEFAULT NULL COMMENT '删除时间', `F_DeleteUserId

# 创建自定义Trainer

In [13]:
# 自定义 Trainer 以跳过设备迁移逻辑
class CustomTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        # 重写此方法，避免 Trainer 将模型强制迁移到 GPU 或其他设备
        pass
    


In [14]:
# 初始化 Trainer
# trainer = CustomTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     tokenizer=tokenizer,
# )


# 初始化 预处理后的数据进行Trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    tokenizer=tokenizer,
)

Detected kernel version 4.19.24, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


# 开始微调

In [15]:
trainer.train()

[2024-11-21 14:09:32,468] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile
collect2: error: ld returned 1 exit status
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


RuntimeError: CUDA out of memory. Tried to allocate 112.00 MiB (GPU 0; 22.20 GiB total capacity; 20.73 GiB already allocated; 86.12 MiB free; 21.05 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

# 保存微调后的模型

In [ ]:
model.save_pretrained(train_model_name)
tokenizer.save_pretrained(train_model_name)

In [ ]:
# !export PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:128